## **Import Required libraries**

In [0]:
import requests
import json
import os
from datetime import datetime,timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

## **API Details**

In [0]:
api_key = #<YOUR API KEY>"
base_url = "https://api.themoviedb.org"

##_API LIST_##
#3/discover/movie - done
#3/movie/{id}
#3/movie/{id}/credits
#3/genre/movie/list
#3/movie/{id}/videos

## **Get the load dates**

In [0]:

start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")

#start_date = '2024-01-01'#audit[-1]["max_release_date"] # To get last max_release_date
#end_date = '2024-02-28'#(datetime.today() - timedelta(days=100)).strftime("%Y-%m-%d")

print(start_date, end_date)

2024-01-01 2024-02-28


## **Session**

In [0]:
session = requests.Session()

## **Get Page Count**

In [0]:
url_movies = f"{base_url}/3/discover/movie"


params = {
    "api_key": api_key,
    "primary_release_date.gte": start_date,
    "primary_release_date.lte": end_date ,
    "page": 1
}

#get the page count

page_Count = session.get(url_movies, params=params).json()['total_pages']

print(page_Count)

380


## **Thread Conf**

In [0]:
page_count = page_Count 
max_workers_pages = 5
max_workers_movies = 20

## **Path SetUp**

In [0]:
spark.sql('USE movie_project.bronze')

DataFrame[]

In [0]:
today = datetime.today()
year = today.strftime("%Y")
month = today.strftime("%m")
date_str = today.strftime("%Y-%m-%d")

volume_path  = f"/Volumes/movie_project/bronze/movies/{year}/{month}"
volume_path_details = f"/Volumes/movie_project/bronze/movies_extended/{year}/{month}"
volume_path_credits = f"/Volumes/movie_project/bronze/credits/{year}/{month}"

dbutils.fs.mkdirs(volume_path)
dbutils.fs.mkdirs(volume_path_details)
dbutils.fs.mkdirs(volume_path_credits)

print('Volume paths created')

final_file_path = f"{volume_path}/movies_{date_str}.json"
final_file_path_details = f"{volume_path_details}/movies_details_{date_str}.json"
final_file_path_credits = f"{volume_path_credits}/credits_{date_str}.json"

temp_file_path = final_file_path + ".tmp"
temp_file_path_details = final_file_path_details + ".tmp"
temp_file_path_credits = final_file_path_credits + ".tmp"

Volume paths created


## **Functions**

In [0]:
def fetch_page(page):
    try:
        params['page']=page
        response = session.get(url_movies, params=params).json()
        return response.get("results", [])
    except:
        return []

In [0]:
def fetch_movie_data(movie):
    try:
        movie_id = movie["id"]

        url_details = f"{base_url}/3/movie/{movie_id}"
        url_credits = f"{base_url}/3/movie/{movie_id}/credits"

        details = session.get(url_details, params={"api_key": api_key}).json()
        credits = session.get(url_credits, params={"api_key": api_key}).json()

        return movie, details, credits
    except:
        return None, None, None


## **Ingestion Process - Movies**

In [0]:
#Write data

def write_to_file(file_path,data):
    file_path = file_path 
    data = data
    dbutils.fs.put(file_path, json.dumps(data) + "\n", overwrite=True)


In [0]:
try:
    all_movies = []

    # STEP 1: FETCH ALL PAGES IN PARALLEL
    with ThreadPoolExecutor(max_workers=max_workers_pages) as executor:
        futures = [executor.submit(fetch_page, i) for i in range(1, page_count + 1)]

        for future in as_completed(futures):
            results = future.result()
            all_movies.extend(results)

    print(f"Total movies fetched: {len(all_movies)}")

    # STEP 2: FETCH DETAILS + CREDITS IN PARALLEL
    movie_list = []
    details_list = []
    credits_list = []

    with ThreadPoolExecutor(max_workers=max_workers_movies) as executor:
        futures = [executor.submit(fetch_movie_data, m) for m in all_movies]

        for future in as_completed(futures):
            movie_data, details_data, credits_data = future.result()
            if movie_data:
                movie_list.append(movie_data)
            if details_data:
                details_list.append(details_data)
            if credits_data:
                credits_list.append(credits_data)
    all_details = {temp_file_path:movie_list,temp_file_path_details: details_list,temp_file_path_credits: credits_list}

    #write data into files
    with ThreadPoolExecutor(max_workers=3) as executor:
        futures = [executor.submit(write_to_file, i,j) for i,j in all_details.items()]

    print(f"Total movie details fetched: {len(movie_list)}")

    #  STEP 4: ATOMIC RENAME
    
    dbutils.fs.mv(temp_file_path, final_file_path)
    dbutils.fs.mv(temp_file_path_details, final_file_path_details)
    dbutils.fs.mv(temp_file_path_credits, final_file_path_credits)

    print("Data ingestion completed successfully")

except Exception as e:
    print("Error:", e)
    for path in [temp_file_path, temp_file_path_details, temp_file_path_credits]:
        try:
            dbutils.fs.ls(path)
            dbutils.fs.rm(path, True)
        except Exception:
            print(f"{path} does not exist")

            

Total movies fetched: 7587
Wrote 4362054 bytes.
Wrote 7852933 bytes.
Wrote 27563864 bytes.
Total movie details fetched: 7587
Data ingestion completed successfully


## **End Session**

In [0]:
session.close()